# **<font color="red">Multi-Speaker TTS</font>**
- For multi-speaker audio, you will need a `MultiSpeakerVoiceConfig` object with each speaker (up to 2) configured as a `SpeakerVoiceConfig`. We need to define each `speaker` with same names used in the prompt.

In [ ]:
import os
import asyncio
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
import google.genai.types as types
from config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = "gemini-2.5-flash"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Create LlmAgent (chatbot)
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="ChatAgent",
    instruction="""
    You are a helpful assistant. Respond politely and concisely to user questions.
    """
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# Create a new session
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# -----------------------------
# Setup Runner
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service
)

# -----------------------------
# Chat function
# -----------------------------
async def chat(user_input):
    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )
    
    events = runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
    
    async for event in events:
        # Print final response only
        if event.is_final_response() and event.content and event.content.parts:
            text = event.content.parts[0].text
            print("ChatAgent:", text)

# -----------------------------
# Chat
# -----------------------------
await chat("Hello! Can you suggest a fun weekend activity?")
await chat("Give me a short summary of Artificial Intelligence.")


In [ ]:
from google import genai
from google.genai import types
import wave

# Set up the wave file to save the output:
def wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
   with wave.open(filename, "wb") as wf:
      wf.setnchannels(channels)
      wf.setsampwidth(sample_width)
      wf.setframerate(rate)
      wf.writeframes(pcm)

client = genai.Client()

prompt = """TTS the following conversation between Joe and Jane:
         Joe: How's it going today Jane?
         Jane: Not too bad, how about you?"""

response = client.models.generate_content(
   model="gemini-2.5-flash-preview-tts",
   contents=prompt,
   config=types.GenerateContentConfig(
      response_modalities=["AUDIO"],
      speech_config=types.SpeechConfig(
         multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
            speaker_voice_configs=[
               types.SpeakerVoiceConfig(
                  speaker='Joe',
                  voice_config=types.VoiceConfig(
                     prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Kore',
                     )
                  )
               ),
               types.SpeakerVoiceConfig(
                  speaker='Jane',
                  voice_config=types.VoiceConfig(
                     prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Puck',
                     )
                  )
               ),
            ]
         )
      )
   )
)

data = response.candidates[0].content.parts[0].inline_data.data

file_name='out.wav'
wave_file(file_name, data) # Saves the file to current directory

In [2]:
"""
Realistic Multi-Speaker Research → Discussion → Translation → TTS Pipeline

Speakers:
  1. Aarav  (Sadaltager - Analytical, structured)
  2. Kabir  (Sulafat - Warm, reflective)

Flow:
1. Research Agent
2. Real Discussion Agent
3. Translation Agent (optional)
4. Multi-Speaker TTS → WAV file
"""

import os
import asyncio
import wave
from datetime import datetime

from google import genai
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.tools.google_search_tool import google_search

from config import config

# ============================================================
# CONFIG
# ============================================================

APP_NAME = "real_discussion_tts_app"
USER_ID = "user_001"

SESSION_RESEARCH = "session_research"
SESSION_DISCUSS  = "session_discussion"
SESSION_TRANS    = "session_translation"

MODEL = "gemini-2.5-flash"
TTS_MODEL = "gemini-2.5-flash-preview-tts"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# ============================================================
# AGENTS
# ============================================================

# 1️⃣ Research Agent
research_agent = LlmAgent(
    name="research_agent",
    model=MODEL,
    instruction="""
You are a deep research analyst.
Research the given topic thoroughly using Google search.
Provide a structured, insightful research summary.
""",
    tools=[google_search],
)

# 2️⃣ Real Discussion Agent
discussion_agent = LlmAgent(
    name="discussion_agent",
    model=MODEL,
    instruction="""
You are a professional podcast script writer.

Create a NATURAL, REALISTIC conversation between:

- Aarav (analytical, structured thinker)
- Kabir (warm, curious, reflective)

IMPORTANT RULES:

1. They MUST address each other by name naturally.
2. They MUST react to what the other just said.
3. They MUST ask follow-up questions.
4. Add mild agreement/disagreement.
5. Make it feel like a real podcast discussion.
6. No long monologues.
7. Keep it dynamic and engaging.
8. 3–4 minutes long.

STRICT FORMAT:

Aarav: ...
Kabir: ...
Aarav: ...
Kabir: ...

Do NOT add narration.
Do NOT add stage directions.
Return only the conversation.
""",
)

# 3️⃣ Translation Agent
translation_agent = LlmAgent(
    name="translation_agent",
    model=MODEL,
    instruction="""
You are a professional translator.

Translate the provided conversation into the target language.
Preserve speaker labels exactly:

Aarav:
Kabir:

Do not add commentary.
Return only translated conversation.
""",
)

# ============================================================
# SERVICES
# ============================================================

session_service = InMemorySessionService()

# ============================================================
# HELPER: Run Agent
# ============================================================

async def run_agent(agent, session_id, message):

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id
    )

    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=message)]
    )

    output = ""

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=content
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    output += part.text

    return output.strip()

# ============================================================
# TTS FUNCTION
# ============================================================

def save_wave(filename, pcm, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)


def generate_multi_speaker_tts(conversation_text):

    client = genai.Client()

    response = client.models.generate_content(
        model=TTS_MODEL,
        contents=conversation_text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                    speaker_voice_configs=[
                        types.SpeakerVoiceConfig(
                            speaker="Aarav",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                                    voice_name="Charon"
                                )
                            )
                        ),
                        types.SpeakerVoiceConfig(
                            speaker="Kabir",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                                    voice_name="Puck"
                                )
                            )
                        ),
                    ]
                )
            )
        )
    )

    audio_data = response.candidates[0].content.parts[0].inline_data.data

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"real_discussion_{timestamp}.wav"

    save_wave(filename, audio_data)

    print(f"\n✅ Audio saved as: {filename}")

# ============================================================
# MAIN PIPELINE
# ============================================================

async def run_pipeline(topic, target_language="English"):

    print("\n🔎 STEP 1: Researching Topic...\n")
    research_output = await run_agent(
        research_agent,
        SESSION_RESEARCH,
        f"Research this topic deeply:\n\n{topic}"
    )

    print("✅ Research Completed\n")

    print("🗣 STEP 2: Generating Real Discussion...\n")
    discussion_output = await run_agent(
        discussion_agent,
        SESSION_DISCUSS,
        f"Based on this research:\n\n{research_output}\n\nCreate a realistic podcast-style discussion."
    )

    print("✅ Discussion Generated\n")

    if target_language.lower() != "english":
        print("🌐 STEP 3: Translating Discussion...\n")

        discussion_output = await run_agent(
            translation_agent,
            SESSION_TRANS,
            f"Translate to {target_language}:\n\n{discussion_output}"
        )

        print("✅ Translation Completed\n")

    print("🔊 STEP 4: Generating Multi-Speaker TTS...\n")
    generate_multi_speaker_tts(discussion_output)

    print("\n🎉 Pipeline Completed Successfully!\n")

# ============================================================
# RUN (Jupyter)
# ============================================================

await run_pipeline(
    topic="AI with Quantum Power",
    target_language="English"
)



🔎 STEP 1: Researching Topic...

✅ Research Completed

🗣 STEP 2: Generating Real Discussion...

✅ Discussion Generated

🔊 STEP 4: Generating Multi-Speaker TTS...


✅ Audio saved as: real_discussion_20260225_161008.wav

🎉 Pipeline Completed Successfully!



In [12]:
"""
Realistic Multi-Speaker Research → Discussion → Translation → TTS Pipeline

Speakers:
  1. Vikas  (Charon - Analytical, structured)
  2. Rohit  (Puck - Warm, reflective)
"""

import os
import asyncio
import wave
from datetime import datetime

from google import genai
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.tools.google_search_tool import google_search

from config import config

# ============================================================
# CONFIG
# ============================================================

APP_NAME = "real_discussion_tts_app"
USER_ID = "user_001"

SESSION_RESEARCH = "session_research"
SESSION_DISCUSS  = "session_discussion"
SESSION_TRANS    = "session_translation"

MODEL = "gemini-2.5-flash"
TTS_MODEL = "gemini-2.5-flash-preview-tts"

OUTPUT_DIR = "generated_discussion"
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# ============================================================
# AGENTS
# ============================================================

research_agent = LlmAgent(
    name="research_agent",
    model=MODEL,
    instruction="""
You are a deep research analyst.
Research the given topic thoroughly using Google search.
Produce a structured, insightful research summary.
Length: 3000–4000 words.
Include facts, statistics, examples, and references where possible.
Output only the research content.
"""
,
    tools=[google_search],
)

discussion_agent = LlmAgent(
    name="discussion_agent",
    model=MODEL,
    instruction="""
You are a professional podcast script writer.

Create a NATURAL, DEEP, REALISTIC conversation between:

- Vikas (analytical, structured thinker)
- Rohit (warm, curious, reflective)

Use the research content provided to produce insightful discussion.
Rules:
1. Address each other by name naturally.
2. React to what the other just said.
3. Ask follow-up questions.
4. Add mild agreement/disagreement.
5. Keep exchanges short; no long monologues.
6. Make it dynamic and engaging.
7. Duration: 2000–3000 words.
8. STRICT FORMAT:

Vikas: ...
Rohit: ...
Vikas: ...
Rohit: ...

Do NOT add narration or stage directions.
Do NOT summarize or shorten.
Use realistic phrasing and everyday language.

Example style:

Vikas: I’m telling you, Rohit, quantum computing is like unlocking a new universe of possibilities for AI.  
Rohit: I hear you, but how do we know it will actually outperform classical computers?  
Vikas: Well, consider the speed and parallelism. Tasks that would take years might take minutes.  
Rohit: That’s impressive, but I wonder about error rates and stability…
Return only the conversation.
"""
)

translation_agent = LlmAgent(
    name="translation_agent",
    model=MODEL,
    instruction="""
You are a professional translator.

Translate the provided conversation into the target language.
Preserve speaker labels exactly:

Vikas:
Rohit:

Do not add commentary.
Return only translated conversation.
"""
)

# ============================================================
# SERVICES
# ============================================================

session_service = InMemorySessionService()

# ============================================================
# HELPER: Run Agent
# ============================================================

async def run_agent(agent, session_id, message):
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id
    )

    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=message)]
    )

    output = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=content
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    output += part.text

    return output.strip()

# ============================================================
# TTS FUNCTION
# ============================================================

def save_wave(filename, pcm, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

def generate_multi_speaker_tts(topic_name, conversation_text):

    client = genai.Client()

    response = client.models.generate_content(
        model=TTS_MODEL,
        contents=conversation_text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                    speaker_voice_configs=[
                        types.SpeakerVoiceConfig(
                            speaker="Vikas",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                                    voice_name="Charon"
                                )
                            )
                        ),
                        types.SpeakerVoiceConfig(
                            speaker="Rohit",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                                    voice_name="Puck"
                                )
                            )
                        ),
                    ]
                )
            )
        )
    )

    audio_data = response.candidates[0].content.parts[0].inline_data.data
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(OUTPUT_DIR, f"{topic_name}_{timestamp}.wav")

    save_wave(filename, audio_data)
    print(f"\n✅ Audio saved: {filename}")

# ============================================================
# MAIN PIPELINE
# ============================================================

async def run_pipeline(topic, target_language="English"):

    print("\n🔎 STEP 1: Researching Topic...\n")
    research_output = await run_agent(
        research_agent,
        SESSION_RESEARCH,
        f"Research this topic deeply:\n\n{topic}"
    )
    word_count = len(research_output.split())
    print(f"✅ Research Completed ({word_count} words)\n")
    print("📄 Full Research Content:\n")
    print(research_output)

    print("\n🗣 STEP 2: Generating Real Discussion...\n")
    discussion_output = await run_agent(
        discussion_agent,
        SESSION_DISCUSS,
        f"Based on this research:\n\n{research_output}\n\nCreate a realistic, deep podcast-style discussion."
    )
    discussion_word_count = len(discussion_output.split())
    print(f"✅ Discussion Generated ({discussion_word_count} words)\n")
    print("💬 Full Discussion:\n")
    print(discussion_output)

    if target_language.lower() != "english":
        print("\n🌐 STEP 3: Translating Discussion...\n")
        translated_output = await run_agent(
            translation_agent,
            SESSION_TRANS,
            f"Translate to {target_language}:\n\n{discussion_output}"
        )
        print("✅ Translation Completed\n")
        print("💬 Full Translated Discussion:\n")
        print(translated_output)
        discussion_output = translated_output  # pass translated content to TTS

    print("\n🔊 STEP 4: Generating Multi-Speaker TTS...\n")
    generate_multi_speaker_tts(topic, discussion_output)

    print("\n🎉 Pipeline Completed Successfully!\n")

# ============================================================
# RUN EXAMPLE (Jupyter or Python)
# ============================================================

await run_pipeline(
    topic="AI with Quantum Power",
    target_language="English"
)



🔎 STEP 1: Researching Topic...

✅ Research Completed (1951 words)

📄 Full Research Content:

## AI with Quantum Power: A Paradigm Shift in Computational Intelligence

The convergence of Artificial Intelligence (AI) and Quantum Computing (QC), often termed Quantum AI, represents a transformative frontier in computational science, promising to unlock capabilities far exceeding the limits of classical computing. This powerful synergy envisions an era where AI models can process information with unprecedented speed, accuracy, and efficiency, tackling complex problems that are currently intractable. While still in its nascent stages, Quantum AI is poised to revolutionize industries, accelerate scientific discovery, and fundamentally redefine our understanding of intelligent systems.

### Understanding Quantum AI: A Synergistic Relationship

At its core, Quantum AI integrates the principles of quantum mechanics with existing machine learning and AI concepts. Classical AI systems rely on bin

In [16]:
"""
Realistic Multi-Speaker Research → Discussion → Translation → TTS Pipeline

Speakers:
  1. Vikas  (Charon - Analytical, structured)
  2. Rohit  (Puck - Warm, reflective)
"""

import os
import asyncio
import wave
from datetime import datetime

from google import genai
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.tools.google_search_tool import google_search

from config import config

# ============================================================
# CONFIG
# ============================================================

APP_NAME = "real_discussion_tts_app"
USER_ID = "user_001"

SESSION_RESEARCH = "session_research"
SESSION_DISCUSS  = "session_discussion"
SESSION_TRANS    = "session_translation"

MODEL = "gemini-2.5-flash"
TTS_MODEL = "gemini-2.5-flash-preview-tts"

OUTPUT_DIR = "generated_discussion"
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# ============================================================
# AGENTS
# ============================================================

research_agent = LlmAgent(
    name="research_agent",
    model=MODEL,
    instruction="""
You are a deep research analyst.
Research the given topic thoroughly using Google search.
Produce a structured, insightful research summary.
Length: 2000–3000 words.
Include facts, statistics, examples, and references where possible.
Output only the research content.
"""
,
    tools=[google_search],
)

discussion_agent = LlmAgent(
    name="discussion_agent",
    model=MODEL,
    instruction="""
You are a professional podcast script writer.

Create a NATURAL, DEEP, REALISTIC conversation between:

- Vikas (analytical, structured thinker)
- Rohit (warm, curious, reflective)

Use the research content provided to produce insightful discussion.
Rules:
1. Address each other by name naturally.
2. React to what the other just said.
3. Ask follow-up questions.
4. Add mild agreement/disagreement.
5. Keep exchanges short; no long monologues.
6. Make it dynamic and engaging.
7. Duration: 1500–2500 words.
8. STRICT FORMAT:

Vikas: ...
Rohit: ...
Vikas: ...
Rohit: ...

Do NOT add narration or stage directions.
Do NOT summarize or shorten.
Use realistic phrasing and everyday language.

Example style:

Vikas: I’m telling you, Rohit, quantum computing is like unlocking a new universe of possibilities for AI.  
Rohit: I hear you, but how do we know it will actually outperform classical computers?  
Vikas: Well, consider the speed and parallelism. Tasks that would take years might take minutes.  
Rohit: That’s impressive, but I wonder about error rates and stability…
Return only the conversation.
"""
)

translation_agent = LlmAgent(
    name="translation_agent",
    model=MODEL,
    instruction="""
You are a professional translator.

Translate the provided conversation into the target language.
Preserve speaker labels exactly:

Vikas:
Rohit:

Do not add commentary.
Return only translated conversation.
"""
)

# ============================================================
# SERVICES
# ============================================================

session_service = InMemorySessionService()

# ============================================================
# HELPER: Run Agent
# ============================================================

async def run_agent(agent, session_id, message):
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id
    )

    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=message)]
    )

    output = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=content
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    output += part.text

    return output.strip()

# ============================================================
# TTS FUNCTION
# ============================================================

def save_wave(filename, pcm, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

def generate_multi_speaker_tts(topic_name, conversation_text):

    client = genai.Client()

    response = client.models.generate_content(
        model=TTS_MODEL,
        contents=conversation_text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                    speaker_voice_configs=[
                        types.SpeakerVoiceConfig(
                            speaker="Vikas",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                                    voice_name="Charon"
                                )
                            )
                        ),
                        types.SpeakerVoiceConfig(
                            speaker="Rohit",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                                    voice_name="Puck"
                                )
                            )
                        ),
                    ]
                )
            )
        )
    )

    audio_data = response.candidates[0].content.parts[0].inline_data.data
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(OUTPUT_DIR, f"{topic_name}_{timestamp}.wav")

    save_wave(filename, audio_data)
    print(f"\n✅ Audio saved: {filename}")

# ============================================================
# MAIN PIPELINE
# ============================================================

async def run_pipeline(topic, target_language="English"):

    print("\n🔎 STEP 1: Researching Topic...\n")
    research_output = await run_agent(
        research_agent,
        SESSION_RESEARCH,
        f"Research this topic deeply:\n\n{topic}"
    )
    word_count = len(research_output.split())
    print(f"✅ Research Completed ({word_count} words)\n")
    print("📄 Full Research Content:\n")
    print(research_output)

    print("\n🗣 STEP 2: Generating Real Discussion...\n")
    discussion_output = await run_agent(
        discussion_agent,
        SESSION_DISCUSS,
        f"Based on this research:\n\n{research_output}\n\nCreate a realistic, deep podcast-style discussion."
    )
    discussion_word_count = len(discussion_output.split())
    print(f"✅ Discussion Generated ({discussion_word_count} words)\n")
    print("💬 Full Discussion:\n")
    print(discussion_output)

    if target_language.lower() != "english":
        print("\n🌐 STEP 3: Translating Discussion...\n")
        translated_output = await run_agent(
            translation_agent,
            SESSION_TRANS,
            f"Translate to {target_language}:\n\n{discussion_output}"
        )
        print("✅ Translation Completed\n")
        print("💬 Full Translated Discussion:\n")
        print(translated_output)
        discussion_output = translated_output  # pass translated content to TTS

    print("\n🔊 STEP 4: Generating Multi-Speaker TTS...\n")
    generate_multi_speaker_tts(topic, discussion_output)

    print("\n🎉 Pipeline Completed Successfully!\n")

# ============================================================
# RUN EXAMPLE (Jupyter or Python)
# ============================================================

await run_pipeline(
    topic="Agentic AI in India",
    target_language="Hindi"
)



🔎 STEP 1: Researching Topic...

✅ Research Completed (2091 words)

📄 Full Research Content:

## Agentic AI in India: A Transformative Leap Towards Autonomous Intelligence

India is rapidly emerging as a significant player in the global Agentic AI landscape, demonstrating a profound shift towards autonomous AI systems capable of independent decision-making and goal-oriented actions. With over 80% of Indian organizations actively exploring the development of autonomous agents, and 24% already deploying them, the nation is witnessing a substantial transformation across various sectors. This widespread adoption is driven by a strong desire for automation, multi-agent workflows, and the tangible business benefits being realized, including enhanced productivity and a positive return on investment (ROI).

### The Evolving Landscape of Agentic AI in India

Agentic AI systems, unlike traditional AI that primarily responds to predefined instructions, are designed to autonomously perceive, reaso

In [1]:
"""
Realistic Multi-Speaker Research → Discussion → Translation → TTS Pipeline

Speakers:
  1. Vikas  (Charon - Analytical, structured)
  2. Rohit  (Puck - Warm, reflective)
"""

import os
import asyncio
import wave
from datetime import datetime

from google import genai
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.tools.google_search_tool import google_search

from config import config

# ============================================================
# CONFIG
# ============================================================

APP_NAME = "real_discussion_tts_app"
USER_ID = "user_001"

SESSION_RESEARCH = "session_research"
SESSION_DISCUSS  = "session_discussion"
SESSION_TRANS    = "session_translation"

MODEL = "gemini-2.5-flash"
TTS_MODEL = "gemini-2.5-flash-preview-tts"

OUTPUT_DIR = "generated_discussion"
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# ============================================================
# AGENTS
# ============================================================

research_agent = LlmAgent(
    name="research_agent",
    model=MODEL,
    instruction="""
You are a deep research analyst.
Research the given topic thoroughly using Google search.
Produce a structured, insightful research summary.
Length: 2000–3000 words.
Include facts, statistics, examples, and references where possible.
Output only the research content.
"""
,
    tools=[google_search],
)

discussion_agent = LlmAgent(
    name="discussion_agent",
    model=MODEL,
    instruction="""
You are a professional podcast script writer.

Create a NATURAL, DEEP, REALISTIC conversation between:

- Vikas (analytical, structured thinker)
- Rohit (warm, curious, reflective)

Use the research content provided to produce insightful discussion.
Rules:
1. Address each other by name naturally.
2. React to what the other just said.
3. Ask follow-up questions.
4. Add mild agreement/disagreement.
5. Keep exchanges short; no long monologues.
6. Make it dynamic and engaging.
7. Duration: 1500–2500 words.
8. STRICT FORMAT:

Vikas: ...
Rohit: ...
Vikas: ...
Rohit: ...

Do NOT add narration or stage directions.
Do NOT summarize or shorten.
Use realistic phrasing and everyday language.

Example style:

Vikas: I’m telling you, Rohit, quantum computing is like unlocking a new universe of possibilities for AI.  
Rohit: I hear you, but how do we know it will actually outperform classical computers?  
Vikas: Well, consider the speed and parallelism. Tasks that would take years might take minutes.  
Rohit: That’s impressive, but I wonder about error rates and stability…
Return only the conversation.
"""
)

translation_agent = LlmAgent(
    name="translation_agent",
    model=MODEL,
    instruction="""
You are a professional translator.

Translate the provided conversation into the target language.
Preserve speaker labels exactly:

Vikas:
Rohit:

Do not add commentary.
Return only translated conversation.
"""
)

# ============================================================
# SERVICES
# ============================================================

session_service = InMemorySessionService()

# ============================================================
# HELPER: Run Agent
# ============================================================

async def run_agent(agent, session_id, message):
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id
    )

    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=message)]
    )

    output = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=content
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    output += part.text

    return output.strip()

# ============================================================
# TTS FUNCTION
# ============================================================

def save_wave(filename, pcm, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

def generate_multi_speaker_tts(topic_name, conversation_text):

    client = genai.Client()

    response = client.models.generate_content(
        model=TTS_MODEL,
        contents=conversation_text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                    speaker_voice_configs=[
                        types.SpeakerVoiceConfig(
                            speaker="Vikas",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                                    voice_name="Charon"
                                )
                            )
                        ),
                        types.SpeakerVoiceConfig(
                            speaker="Rohit",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                                    voice_name="Puck"
                                )
                            )
                        ),
                    ]
                )
            )
        )
    )

    audio_data = response.candidates[0].content.parts[0].inline_data.data
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(OUTPUT_DIR, f"{topic_name}_{timestamp}.wav")

    save_wave(filename, audio_data)
    print(f"\n✅ Audio saved: {filename}")

# ============================================================
# MAIN PIPELINE
# ============================================================

async def run_pipeline(topic, target_language="English"):

    print("\n🔎 STEP 1: Researching Topic...\n")
    research_output = await run_agent(
        research_agent,
        SESSION_RESEARCH,
        f"Research this topic deeply:\n\n{topic}"
    )
    word_count = len(research_output.split())
    print(f"✅ Research Completed ({word_count} words)\n")
    print("📄 Full Research Content:\n")
    print(research_output)

    print("\n🗣 STEP 2: Generating Real Discussion...\n")
    discussion_output = await run_agent(
        discussion_agent,
        SESSION_DISCUSS,
        f"Based on this research:\n\n{research_output}\n\nCreate a realistic, deep podcast-style discussion."
    )
    discussion_word_count = len(discussion_output.split())
    print(f"✅ Discussion Generated ({discussion_word_count} words)\n")
    print("💬 Full Discussion:\n")
    print(discussion_output)

    if target_language.lower() != "english":
        print("\n🌐 STEP 3: Translating Discussion...\n")
        translated_output = await run_agent(
            translation_agent,
            SESSION_TRANS,
            f"Translate to {target_language}:\n\n{discussion_output}"
        )
        print("✅ Translation Completed\n")
        print("💬 Full Translated Discussion:\n")
        print(translated_output)
        discussion_output = translated_output  # pass translated content to TTS

    print("\n🔊 STEP 4: Generating Multi-Speaker TTS...\n")
    generate_multi_speaker_tts(topic, discussion_output)

    print("\n🎉 Pipeline Completed Successfully!\n")

# ============================================================
# RUN EXAMPLE (Jupyter or Python)
# ============================================================

await run_pipeline(
    topic="Discussion on God of Universe Baba Rohit Who lived in Delhi.",
    target_language="Haryanvi Hindi"
)



🔎 STEP 1: Researching Topic...

✅ Research Completed (919 words)

📄 Full Research Content:

Despite extensive research, no widely recognized or prominent spiritual leader explicitly identified as "God of Universe Baba Rohit" who lived in Delhi could be found in the provided search results. The specific title "God of Universe" combined with the name "Baba Rohit" does not appear to correspond to a well-documented public figure in spiritual or religious discourse.

However, the searches did reveal several individuals named "Rohit" who are associated with spirituality, religious practices, or related fields in Delhi or India. It is possible that the inquirer might be referring to one of these individuals, or to a lesser-known figure, or perhaps the title "God of Universe" is a personal designation not widely used in public references.

Here are the details of individuals named Rohit with spiritual connections found in the research:

**1. Sri Guru Rohit Arya:**
Sri Guru Rohit Arya is a not